In [ ]:
!pip3 install beautifulsoup4

In [1]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS

from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx

from dotenv import load_dotenv

from pprint import pprint

import bs4
import re
import os

c:\souce\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\soldesk\AppData\Local\Temp\ipykernel_5244\3062218108.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")

In [3]:
watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
)

In [4]:
web_loader = WebBaseLoader(web_paths=['https://n.news.naver.com/article/037/0000038261?cds=news_media_pc&type=editn'],
                           bs_kwargs=dict(parse_only=bs4.SoupStrainer("article", attrs={"id":"dic_area"})))
web_docs = web_loader.load()

print(f"총 페이지 수 : {len(web_docs)}")
print(web_docs)

총 페이지 수 : 1
[Document(metadata={'source': 'https://n.news.naver.com/article/037/0000038261?cds=news_media_pc&type=editn'}, page_content='\n 젠슨황, 6월 4일 방한해 SK·LG·네이버 총수들과 ‘삼겹살 회동’ 예정\n\n\n\n최태원 SK그룹 회장(오른쪽)은 6월 1일 대만 타이베이에서 젠슨 황 엔비디아 최고경영자와 만나 인공지능(AI) 협력 방안 등에 대해 논의했다.  SK하이닉스 페이스북최태원 SK그룹 회장이 젠슨 황 엔비디아 최고경영자(CEO)와 만나 두 기업의 ‘인공지능(AI) 동맹’을 다시금 과시했다. SK하이닉스는 6월 2일 자사 페이스북 계정에 “SK하이닉스가 시가총액 1조 달러를 달성한 가운데, 최 회장과 황 CEO를 비롯한 양사 경영진이 만나 그 의미를 함께 나눴다”면서 두 사람이 전날 대만에서 열린 ‘GTC 타이베이 2026’ 행사에서 만난 사실을 공개했다. 최 회장과 황 CEO는 AI 메모리 분야에서 함께 이뤄낸 성과를 되새기고 AI 인프라의 새로운 지평을 함께 열어가겠다는 의지를 확인한 것으로 알려졌다. 같은 날 황 CEO는 타이베이의 한 해산물 식당에서 ‘코리아 파트너스 나잇’ 행사를 갖고 곽노정 SK하이닉스 사장, 김재준 삼성전자 디바이스 솔루션(DS) 부문 메모리사업부 부사장, 김유원 네이버클라우드 대표와도 만났다. 이 자리에서 황 CEO는 “한국에는 훌륭한 생태계와 기술력 있는 기업, 연구팀, 과학 커뮤니티가 있다”며 한국 기업들과의 협력 의지를 재천명했다. 그는 SK하이닉스의 시가총액 1조 달러(약 1511조300억 원) 달성에 대해서도 “정말 자랑스럽고 그들의 성공이 기쁘다”고 언급했다. 지난해 10월 이재용 삼성전자 회장, 정의선 현대차그룹 회장과의 이른바 ‘깐부 회동’으로 관심을 모았던 황 CEO는 6월 4일 다시 방한해 이튿날 최 회장과 구광모 LG그룹 회장, 이해진 네이버 의장 등과 만날 것으로 알려졌다. 이 자

In [5]:
pprint(web_docs[0].page_content)
pprint(web_docs[0].metadata)


('\n'
 ' 젠슨황, 6월 4일 방한해 SK·LG·네이버 총수들과 ‘삼겹살 회동’ 예정\n'
 '\n'
 '\n'
 '\n'
 '최태원 SK그룹 회장(오른쪽)은 6월 1일 대만 타이베이에서 젠슨 황 엔비디아 최고경영자와 만나 인공지능(AI) 협력 방안 등에 대해 '
 '논의했다.  SK하이닉스 페이스북최태원 SK그룹 회장이 젠슨 황 엔비디아 최고경영자(CEO)와 만나 두 기업의 ‘인공지능(AI) 동맹’을 '
 '다시금 과시했다. SK하이닉스는 6월 2일 자사 페이스북 계정에 “SK하이닉스가 시가총액 1조 달러를 달성한 가운데, 최 회장과 황 '
 'CEO를 비롯한 양사 경영진이 만나 그 의미를 함께 나눴다”면서 두 사람이 전날 대만에서 열린 ‘GTC 타이베이 2026’ 행사에서 만난 '
 '사실을 공개했다. 최 회장과 황 CEO는 AI 메모리 분야에서 함께 이뤄낸 성과를 되새기고 AI 인프라의 새로운 지평을 함께 열어가겠다는 '
 '의지를 확인한 것으로 알려졌다. 같은 날 황 CEO는 타이베이의 한 해산물 식당에서 ‘코리아 파트너스 나잇’ 행사를 갖고 곽노정 '
 'SK하이닉스 사장, 김재준 삼성전자 디바이스 솔루션(DS) 부문 메모리사업부 부사장, 김유원 네이버클라우드 대표와도 만났다. 이 자리에서 '
 '황 CEO는 “한국에는 훌륭한 생태계와 기술력 있는 기업, 연구팀, 과학 커뮤니티가 있다”며 한국 기업들과의 협력 의지를 재천명했다. '
 '그는 SK하이닉스의 시가총액 1조 달러(약 1511조300억 원) 달성에 대해서도 “정말 자랑스럽고 그들의 성공이 기쁘다”고 언급했다. '
 '지난해 10월 이재용 삼성전자 회장, 정의선 현대차그룹 회장과의 이른바 ‘깐부 회동’으로 관심을 모았던 황 CEO는 6월 4일 다시 '
 '방한해 이튿날 최 회장과 구광모 LG그룹 회장, 이해진 네이버 의장 등과 만날 것으로 알려졌다. 이 자리에 정 회장도 참석을 검토중이며, '
 '이 회장은 해외 일정상 이번 회동에는 불참할 것으로 전해졌다. 재계에선 황 CEO 방한을 계

In [6]:
content = web_docs[0].page_content.strip()

content

'젠슨황, 6월 4일 방한해 SK·LG·네이버 총수들과 ‘삼겹살 회동’ 예정\n\n\n\n최태원 SK그룹 회장(오른쪽)은 6월 1일 대만 타이베이에서 젠슨 황 엔비디아 최고경영자와 만나 인공지능(AI) 협력 방안 등에 대해 논의했다.  SK하이닉스 페이스북최태원 SK그룹 회장이 젠슨 황 엔비디아 최고경영자(CEO)와 만나 두 기업의 ‘인공지능(AI) 동맹’을 다시금 과시했다. SK하이닉스는 6월 2일 자사 페이스북 계정에 “SK하이닉스가 시가총액 1조 달러를 달성한 가운데, 최 회장과 황 CEO를 비롯한 양사 경영진이 만나 그 의미를 함께 나눴다”면서 두 사람이 전날 대만에서 열린 ‘GTC 타이베이 2026’ 행사에서 만난 사실을 공개했다. 최 회장과 황 CEO는 AI 메모리 분야에서 함께 이뤄낸 성과를 되새기고 AI 인프라의 새로운 지평을 함께 열어가겠다는 의지를 확인한 것으로 알려졌다. 같은 날 황 CEO는 타이베이의 한 해산물 식당에서 ‘코리아 파트너스 나잇’ 행사를 갖고 곽노정 SK하이닉스 사장, 김재준 삼성전자 디바이스 솔루션(DS) 부문 메모리사업부 부사장, 김유원 네이버클라우드 대표와도 만났다. 이 자리에서 황 CEO는 “한국에는 훌륭한 생태계와 기술력 있는 기업, 연구팀, 과학 커뮤니티가 있다”며 한국 기업들과의 협력 의지를 재천명했다. 그는 SK하이닉스의 시가총액 1조 달러(약 1511조300억 원) 달성에 대해서도 “정말 자랑스럽고 그들의 성공이 기쁘다”고 언급했다. 지난해 10월 이재용 삼성전자 회장, 정의선 현대차그룹 회장과의 이른바 ‘깐부 회동’으로 관심을 모았던 황 CEO는 6월 4일 다시 방한해 이튿날 최 회장과 구광모 LG그룹 회장, 이해진 네이버 의장 등과 만날 것으로 알려졌다. 이 자리에 정 회장도 참석을 검토중이며, 이 회장은 해외 일정상 이번 회동에는 불참할 것으로 전해졌다. 재계에선 황 CEO 방한을 계기로 엔비디아와 국내 기업들의 협력이 로보틱스, 피지컬 AI 분야 등으로 확대될 것이라는 전망이 나온다. 황 CEO와 국내

In [7]:
lines = [line  for line in  content.split("\n") if line.strip()]
title = lines[0]

# [.........] 사진 설명 제거 / 당신의 ~~

# page_content 업데이트
# metadata 업데이트

web_docs[0].page_content = "\n\n".join(lines[0:2])
web_docs[0].metadata['title'] = title

In [8]:
pprint(web_docs[0].page_content)
pprint(web_docs[0].metadata)

('젠슨황, 6월 4일 방한해 SK·LG·네이버 총수들과 ‘삼겹살 회동’ 예정\n'
 '\n'
 '최태원 SK그룹 회장(오른쪽)은 6월 1일 대만 타이베이에서 젠슨 황 엔비디아 최고경영자와 만나 인공지능(AI) 협력 방안 등에 대해 '
 '논의했다.  SK하이닉스 페이스북최태원 SK그룹 회장이 젠슨 황 엔비디아 최고경영자(CEO)와 만나 두 기업의 ‘인공지능(AI) 동맹’을 '
 '다시금 과시했다. SK하이닉스는 6월 2일 자사 페이스북 계정에 “SK하이닉스가 시가총액 1조 달러를 달성한 가운데, 최 회장과 황 '
 'CEO를 비롯한 양사 경영진이 만나 그 의미를 함께 나눴다”면서 두 사람이 전날 대만에서 열린 ‘GTC 타이베이 2026’ 행사에서 만난 '
 '사실을 공개했다. 최 회장과 황 CEO는 AI 메모리 분야에서 함께 이뤄낸 성과를 되새기고 AI 인프라의 새로운 지평을 함께 열어가겠다는 '
 '의지를 확인한 것으로 알려졌다. 같은 날 황 CEO는 타이베이의 한 해산물 식당에서 ‘코리아 파트너스 나잇’ 행사를 갖고 곽노정 '
 'SK하이닉스 사장, 김재준 삼성전자 디바이스 솔루션(DS) 부문 메모리사업부 부사장, 김유원 네이버클라우드 대표와도 만났다. 이 자리에서 '
 '황 CEO는 “한국에는 훌륭한 생태계와 기술력 있는 기업, 연구팀, 과학 커뮤니티가 있다”며 한국 기업들과의 협력 의지를 재천명했다. '
 '그는 SK하이닉스의 시가총액 1조 달러(약 1511조300억 원) 달성에 대해서도 “정말 자랑스럽고 그들의 성공이 기쁘다”고 언급했다. '
 '지난해 10월 이재용 삼성전자 회장, 정의선 현대차그룹 회장과의 이른바 ‘깐부 회동’으로 관심을 모았던 황 CEO는 6월 4일 다시 '
 '방한해 이튿날 최 회장과 구광모 LG그룹 회장, 이해진 네이버 의장 등과 만날 것으로 알려졌다. 이 자리에 정 회장도 참석을 검토중이며, '
 '이 회장은 해외 일정상 이번 회동에는 불참할 것으로 전해졌다. 재계에선 황 CEO 방한을 계기로 엔비디아와 국내 기업들의 협력

In [9]:
# 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(web_docs)

len(chunks)

5

In [10]:
# 벡터스토어

vectorstore = FAISS.from_documents(documents=chunks, embedding=watsonx_enbedding)

# 디버깅용
# vectorstore.similarity_search()

retriever = vectorstore.as_retriever(k=3)

rag_prompt = ChatPromptTemplate.from_template(
"""\
당신은 뉴스 기사 QA 시스템입니다.

규칙:
1. 제공된 기사 내용만 사용하세요.
2. 기사 내용에 없는 정보는 추측하지 말고 '기사에서 확인할 수 없습니다'라고 답하세요.

뉴스 기사:
{context}

질문:
{question}

답변:
"""
)

In [11]:
# 체인생성 질의
def format_docs(docs):
    """Document 객체에서 page_content 추출"""
    return "\n\n".join([d.page_content for d in docs])

chain = {"context": retriever | format_docs,
         "question" : RunnablePassthrough()
         } | rag_prompt | watson_llm | StrOutputParser()


question = "타이베이에서 만난 사람들은 누구인가요?"
response = chain.invoke(question)
print(response)

대만 타이베이에서 만난 사람들은 젠슨 황 엔비디아 최고경영자(CEO)와 SK하이닉스 사장 곽노정, 삼성전자 디바이스 솔루션(DS) 부문 메모리사업부 부사장 김재준, 네이버클라우드 대표 김유원입니다.
